# V2: Planner Agent (Planning Autonomy)

V2 = V1 router + **BM25 retrieval over runbooks** + **plan generation**.

Same CC/CD loop, three metrics this time:

- **Routing accuracy** — does the planner pick the right category? (sanity check that V1 still works inside V2)
- **Retrieval recall@k** — is the correct runbook in the top-K BM25 hits?
- **LLM-as-judge** scores on (groundedness, completeness, actionability) — does the generated plan actually follow the runbook?

In [ ]:
import os, sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')
assert os.environ.get('OPENAI_API_KEY'), 'set OPENAI_API_KEY in ../.env'

import pandas as pd
from src import RouterAgent, RunbookIndex, PlannerAgent, evaluate_planner

test_cases = pd.read_csv('../data/test_cases_v2.csv')
test_cases.head()

## 1. Build the planner

The planner composes:
- An improved V1 router (re-used directly from V1).
- A BM25 index over `data/runbooks/*.md`.
- A planner LLM call that takes the alert, the routed category, and the top-K retrieved runbooks.

In [ ]:
router = RouterAgent(prompt_version='improved')
index = RunbookIndex(runbook_dir='../data/runbooks')
print(f'Indexed {len(index.runbooks)} runbooks:')
for rb in index.runbooks:
    print(f'  {rb.runbook_id:<40s} [{rb.category}] {rb.title}')

In [ ]:
# Quick sanity check: a single end-to-end plan
baseline_planner = PlannerAgent(router=router, index=index, prompt_version='baseline')
sample = test_cases.iloc[0]
plan = baseline_planner.plan(sample.alert_text)
print(f'ALERT: {sample.alert_text}\n')
print(f'Category: {plan.category.value}')
print(f'Primary runbook: {plan.primary_runbook}')
print(f'Top-3 BM25 hits: {[(h.runbook.runbook_id, round(h.score,1)) for h in plan.retrieval_hits]}')
print('Steps:')
for i, step in enumerate(plan.steps, 1):
    print(f'  {i}. {step.step}  -- why: {step.why}')

## 2. Test — baseline planner

Run the full evaluation. The judge call is the most expensive step; disable with `judge=False` for a fast preview.

In [ ]:
baseline_eval = evaluate_planner(baseline_planner, test_cases, judge=True)
print(f'Routing accuracy:       {baseline_eval.routing_accuracy:.2%}')
print(f'Recall@1 (retrieval):   {baseline_eval.recall_at_1:.2%}')
print(f'Recall@k (retrieval):   {baseline_eval.recall_at_k:.2%}')
print(f'Avg groundedness  (1-5): {baseline_eval.avg_groundedness:.2f}')
print(f'Avg completeness  (1-5): {baseline_eval.avg_completeness:.2f}')
print(f'Avg actionability (1-5): {baseline_eval.avg_actionability:.2f}')

## 3. Calibrate — inspect failures

Three kinds of failure to look for:

1. **Routing wrong** — propagates from V1; revisit the router prompt.
2. **Retrieval missed the right runbook** — usually means alert wording uses different vocabulary than the runbook (e.g., "impossible-travel login" vs. "suspicious login"). Fix by adding synonyms to the runbook body or boosting symptom sections.
3. **Plan was generated but not grounded in the runbook** — the LLM invented steps. Fix by tightening the planner system prompt.

In [ ]:
df = baseline_eval.predictions
print('--- routing failures ---')
print(df[~df.category_correct][['id','expected_category','predicted_category','alert_text']])
print('\n--- retrieval failures (right runbook NOT in top-k) ---')
print(df[~df.in_top_k][['id','expected_runbook','alert_text']])
print('\n--- grounded-but-low-judge failures ---')
print(df[(df.groundedness < 4) & (df.in_top_k)][['id','groundedness','completeness','actionability','judge_notes']])

## 4. Deploy — improved planner

In [ ]:
improved_planner = PlannerAgent(router=router, index=index, prompt_version='improved')
improved_eval = evaluate_planner(improved_planner, test_cases, judge=True)

import pandas as pd
comparison = pd.DataFrame({
    'metric': ['routing_accuracy','recall@1','recall@k','groundedness','completeness','actionability'],
    'baseline': [baseline_eval.routing_accuracy, baseline_eval.recall_at_1, baseline_eval.recall_at_k,
                 baseline_eval.avg_groundedness, baseline_eval.avg_completeness, baseline_eval.avg_actionability],
    'improved': [improved_eval.routing_accuracy, improved_eval.recall_at_1, improved_eval.recall_at_k,
                 improved_eval.avg_groundedness, improved_eval.avg_completeness, improved_eval.avg_actionability],
})
comparison['delta'] = comparison['improved'] - comparison['baseline']
comparison

## Takeaways

- BM25 alone (no embeddings) gets high recall@k on a small, well-written runbook corpus.
- The biggest planner-quality wins come from a stricter system prompt that requires grounding in the supplied runbook.
- The CC/CD loop is **the same** whether you are tuning a single LLM call or composing multiple components — change one thing at a time, measure on the same eval set, and only ship when the delta is positive.